In [1]:
from pathlib import Path
import subprocess
from collections import Counter

import pandas as pd

In [2]:
data = Path().resolve() / 'resfinder'

In [3]:
cmd = f"""
source activate abricate \
    && bsub \
        -o {data / 'abricate.out'} \
        -e {data / 'abricate.err'} \
            "abricate \
                --db resfinder \
                --nopath \
                --minid 95 \
                --mincov 80 \
                -fofn {data.parent / 'input.txt'} \
                >> {data / 'resfinder.tab'}
            "
"""
subprocess.run(cmd, shell=True)

Job <37427419> is submitted to queue <bio>.


CompletedProcess(args='\nsource activate abricate     && bsub         -o /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/resfinder/abricate.out         -e /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/resfinder/abricate.err             "abricate                 --db resfinder                 --nopath                 --minid 95                 --mincov 80                 -fofn /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/input.txt                 >> /mnt/scratch_dir/lansus/mrsa_cfr_manuscript/resfinder/resfinder.tab\n            "\n', returncode=0)

In [17]:
df = pd.read_table(data / 'resfinder.tab', sep='\t')

In [18]:
cfr_likes = df.groupby('SEQUENCE')['PRODUCT'].apply(lambda x: [y for y in x if 'cfr' in y.lower()]).reset_index(name='cfr-like').explode('cfr-like')

In [19]:
cfr_likes = cfr_likes.drop_duplicates()

In [20]:
merged = df.groupby('SEQUENCE')['PRODUCT'].apply(lambda x: ', '.join(sorted(set(x)))).reset_index(name='resfinder_profile')
merged = merged.drop_duplicates()

In [21]:
merged = merged.merge(cfr_likes, on='SEQUENCE', how='left')

In [22]:
merged.to_csv(data /'resfinder_results.tab', sep='\t')